In [1]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types
from PIL import Image
import base64
import io
import json
from datetime import datetime
import uuid
import time
import re

import brand_director as bd
import nametag_prompts as pn

bd.setup_environment()

# 함수에서 반환된 키를 변수에 저장하여 다른 곳에서 활용
GEMINI_API_KEY, GEMINI_IMAGE_API_KEY = bd.load_gemini_keys()

# 시스템 프롬프트를 변수에 저장하여 다른 곳에서 활용
SYSTEM_PROMPT = pn.SYSTEM_PROMPT

✅ Gemini API 키 로드 완료
✅ Gemini Image API 키 로드 완료


In [2]:
brand_info = {'brand_name': '시스테마',
 'brand_name_en': 'Systema',
 'name_meaning': '체계와 질서를 뜻하는 라틴어 기반의 네이밍',
 'slogan': '데이터로 증명하는 브랜딩의 정석',
 'story_summary': '흩어진 아이디어를 냉철하게 분석하고 논리적 세포 단위로 재구성하여 설득력 있는 브랜드 결과물을 도출합니다.',
 'seed_color': '#4A4A4A',
 'seed_color_reason': '단단한 금속의 질감을 닮은 냉철한 스틸 그레이'}

interview_data_A = "Q1. 시스테마의 '논리적 세포 단위 재구성'이라는 철학을 시각화할 때, 어떤 그래픽 스타일이 가장 적합하다고 생각하시나요?\nA: 정교하게 설계된 그리드 시스템 기반의 미니멀한 고딕 스타일\n\nQ2. 고객이 브랜드로부터 메시지를 받을 때, 시스테마의 페르소나는 어떤 화법을 사용해야 할까요?\nA: 시장의 비효율을 비판하며 명확한 정답을 제시하는 권위적인 어조\n\nQ3. 고객이 시스테마를 처음 접하는 순간, 브랜드의 '정밀함'을 체감할 결정적인 경험은 무엇이어야 할까요?\nA: 모든 정보가 논리적으로 정렬되어 탐색이 극도로 편리한 웹사이트\n\nQ4. 시스테마가 시장에 가장 먼저 출시하여 가치를 증명할 '히어로 제품/서비스'의 형태는 무엇입니까?\nA: 데이터 분석 기반의 브랜드 전략 컨설팅 서비스"

In [3]:
raw_text_B_0 = bd.request_gemini_api(pn.get_B_interview_prompt(brand_info), SYSTEM_PROMPT)
parsed_response_B_0 = bd.parse_ai_response(raw_text_B_0)


----------------------------------------------------------------------------------------------------
💬 AI에 요청 중입니다... 잠시만 기다려주세요.
----------------------------------------------------------------------------------------------------

✅ AI 응답 수신 완료!
응답 텍스트: {
  "required_question_count": 3,
  "reasoning": "브랜드 '시스테마'의 냉철하고 논리적인 이미지에 부합하는 타겟의 깊이 있는 결핍(Pain-...
✅ JSON 파싱 완료!
파싱된 JSON: {"required_question_count": 3, "reasoning": "\ube0c\ub79c\ub4dc '\uc2dc\uc2a4\ud14c\ub9c8'\uc758 \ub0c9\ucca0\ud558\uace0 \ub17c\ub9ac\uc801\uc778 \uc774\ubbf8\uc9c0\uc5d0 \ubd80\ud569\ud558\ub294 \ud0c0\uac9f\uc758 \uae4a\uc774 \uc788\ub294 \uacb0\ud54d(Pain-point)\uacfc \uc2ec\ub9ac\uc801 \ubcf4\uc0c1 \uae30\uc804, \uadf8\ub9ac\uace0 \uadf8\ub4e4\uc774 \uc2e0\ub8b0\ud558\ub294 \uc815\ubcf4 \ud68d\ub4dd \uacbd\ub85c\ub97c \ud30c\uc545\ud558\uc5ec \uc785\uccb4\uc801\uc778 \ud398\ub974\uc18c\ub098\uc640 \uc815\uad50\ud55c \uace0\uac1d \uc5ec\uc815\uc744 \uc124\uacc4\ud558\uace0\uc790 \ud569\ub2c8\ub2e4.", "que

In [4]:
# AI가 왜 이런 질문을 만들었는지 사용자에게 보여주면 신뢰도가 확 올라갑니다!
print(f"💡 AI 분석: {parsed_response_B_0['reasoning']}\n")

# 사용자의 최종 답변을 모아둘 리스트
collected_answers_B = bd.conduct_ai_interview(parsed_response_B_0, title="🤖 Section B 가치 구체화 인터뷰")
interview_data_B = bd.format_interview_responses(collected_answers_B)

💡 AI 분석: 브랜드 '시스테마'의 냉철하고 논리적인 이미지에 부합하는 타겟의 깊이 있는 결핍(Pain-point)과 심리적 보상 기전, 그리고 그들이 신뢰하는 정보 획득 경로를 파악하여 입체적인 페르소나와 정교한 고객 여정을 설계하고자 합니다.


🤖 Section B 가치 구체화 인터뷰
💡 AI 분석: 브랜드 '시스테마'의 냉철하고 논리적인 이미지에 부합하는 타겟의 깊이 있는 결핍(Pain-point)과 심리적 보상 기전, 그리고 그들이 신뢰하는 정보 획득 경로를 파악하여 입체적인 페르소나와 정교한 고객 여정을 설계하고자 합니다.


[질문 1/3. 이 타겟이 침대에 누워 잠들기 직전, 오늘 업무나 일상에서 가장 크게 느꼈을 '결핍'이나 스트레스는 무엇일까요?]
  1. 열심히는 했으나 과정이 뒤섞여 있어 스스로도 성과를 논리적으로 설명하기 힘든 상태
  2. 감각적인 아이디어는 많지만 이를 뒷받침할 구체적인 근거와 데이터가 부족해 설득에 실패한 경험
  3. 비효율적인 시스템과 불필요한 소통으로 인해 정작 중요한 본질에 집중하지 못했다는 자괴감
  4. 주관적인 취향과 감정에 치우친 의사결정들 사이에서 명확한 기준점을 찾지 못한 혼란

[질문 2/3. 이 타겟이 '시스테마'의 서비스를 최종 결제하게 만드는 결정적인 '심리적 트리거'는 무엇일까요?]
  1. 정답이 없는 브랜딩 영역에서 '수치와 논리'라는 명확한 안전장치를 확보했다는 안도감
  2. 아무나 쉽게 흉내 낼 수 없는 전문적이고 정교한 시스템을 소유했다는 지적 우월감
  3. 복잡한 문제를 한 단어로 관통하는 명쾌한 질서를 목격했을 때 느끼는 카타르시스
  4. 결과물의 미학적 가치를 넘어 그 이면의 탄탄한 구조까지 완벽하다는 확신에서 오는 신뢰

[질문 3/3. 이 타겟이 평소 자신의 전문성을 강화하거나 고차원적인 정보를 습득하기 위해 가장 자주 머무는 '디지털/물리적 공간'은 어디인가요?]
  1. 링크드인(LinkedIn)이나 퍼블리(PUBLY)처럼 실무 인사이트와 데이터가 검증된 플랫폼

In [5]:
interview_data_B

"Q1. 이 타겟이 침대에 누워 잠들기 직전, 오늘 업무나 일상에서 가장 크게 느꼈을 '결핍'이나 스트레스는 무엇일까요?\nA: 주관적인 취향과 감정에 치우친 의사결정들 사이에서 명확한 기준점을 찾지 못한 혼란\n\nQ2. 이 타겟이 '시스테마'의 서비스를 최종 결제하게 만드는 결정적인 '심리적 트리거'는 무엇일까요?\nA: 복잡한 문제를 한 단어로 관통하는 명쾌한 질서를 목격했을 때 느끼는 카타르시스\n\nQ3. 이 타겟이 평소 자신의 전문성을 강화하거나 고차원적인 정보를 습득하기 위해 가장 자주 머무는 '디지털/물리적 공간'은 어디인가요?\nA: 업계 최고의 전문가들이 모여 논리적인 비판과 토론을 즐기는 폐쇄적인 커뮤니티"

In [ ]:
raw_text_B_1 = bd.request_gemini_api(pn.get_B1_persona_prompt(brand_info, interview_data_A, interview_data_B), SYSTEM_PROMPT)
parsed_response_B_1 = bd.parse_ai_response(raw_text_B_1)
raw_text_B_2 = bd.request_gemini_api(pn.get_B2_journey_prompt(brand_info, interview_data_A, interview_data_B), SYSTEM_PROMPT)
parsed_response_B_2 = bd.parse_ai_response(raw_text_B_2)


----------------------------------------------------------------------------------------------------
💬 AI에 요청 중입니다... 잠시만 기다려주세요.
----------------------------------------------------------------------------------------------------

⚠️ 서버 과부하 또는 할당량 초과 에러 발생: 503 UNAVAILABLE
⏳ 1분 30초 대기 후 재요청합니다... (재시도 횟수: 1/10)


In [ ]:
parsed_response_B_1, parsed_response_B_2

({'strategic_reasoning': "브랜드 '숲결'의 '지속 가능한 소재의 아름다움'과 고객 인터뷰에서 도출된 '일회용품에 대한 부채감'을 결합할 때, 가장 강력한 구매 전환은 자신의 공간을 도덕적·심미적 완결 상태로 만들고자 하는 '의식 있는 도시 생활자'에게서 발생합니다. 이들은 단순히 예쁜 소품이 아니라, 일상에서 느끼는 환경적 죄책감을 해소해주고 자신의 정체성을 긍정하게 만드는 '정서적 세탁' 기제를 구매하는 것이기에 전환율이 극대화됩니다.",
  'core_target_group': {'definition': '환경적 부채감을 미학적 성취감으로 대체하고자 하는 도시형 컨셔스 테이스트 메이커',
   'demographics': '28-36세 여성 및 남성, 화이트칼라 전문직 또는 크리에이티브 직군, 도심 거주 1인 가구 및 신혼부부',
   'lifestyle_keywords': ['제로웨이스트', '미니멀라이프', '슬로우라이프', '지속가능한인테리어', '컨셔스소비'],
   'consumption_behavior': "배달 음식이나 저가형 패스트 리빙 제품에는 철저히 가성비를 따지지만, 매일 손에 닿고 시선이 머무는 '반영구적 리빙 오브제'에는 10만 원 이상의 가심비를 기꺼이 지불함"},
  'primary_persona': {'name': '김지안',
   'age_and_job': '31세, IT 기업 서비스 기획자',
   'daily_scene': '고강도 업무 후 퇴근하여 문을 열었을 때, 현관에 쌓인 택배 박스와 배달 용기들을 보며 삶이 소모되고 있다는 느낌을 받는 순간',
   'pain_point_and_desire': "세련된 사회적 모습과 달리 일회용품으로 점철된 사적 공간에서의 괴리감. 숲결의 소품을 통해 자신의 일상을 다시 '정갈하고 가치 있는 상태'로 회복하고 싶어 함",
   'emotional_needs': ['죄책감 없는 휴식',
    '윤리적 소비를 통한 자아 효능감',
    '자연에 가까운 질감이 